# Algorithms 17: Tic-Tac-Toe**Learning Objectives:**1. Use Object-Oriented design to encapsulate game state2. Enforce rules via private methods and move validation3. Create Random and Manual Player classes that interact with the Game4. Develop logging and debugging tools for game history**Prerequisites:** Algorithms 01**Estimated time:** 90 minutes**Textbook:** *Justin Skycak — Intro to Algorithms & Machine Learning* Chapter 17

In [ ]:
# Google Colab Setup!pip install -q ipywidgets jupyterquiz ipytest plotly sympy networkx pandas numpy matplotlib tqdmimport json, os, sysfrom pathlib import Pathif Path('/content').exists():    repo_root = Path('/content/upskill')else:    repo_root = Path.cwd()if str(repo_root) not in sys.path:    sys.path.insert(0, str(repo_root))try:    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )except ModuleNotFoundError:    Path('learning_tools.py').write_text('"""Colab-first learning helpers for the interactive math curriculum.\n\nThe functions in this module are intentionally lightweight: they work in a\nplain notebook, in Google Colab, and in local Jupyter without requiring a\nbackend service.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, asdict, field\nfrom datetime import datetime, timedelta, timezone\nimport json\nimport math\nimport os\nimport re\nfrom pathlib import Path\nfrom typing import Any, Callable, Iterable\n\n\nSCHEMA_VERSION = 1\nDEFAULT_PROGRESS_FILENAME = "upskill_math_progress.json"\nDRIVE_PROGRESS_DIR = "MyDrive/upskill"\n\n\ndef _utcnow() -> datetime:\n    return datetime.now(timezone.utc)\n\n\ndef _iso(dt: datetime) -> str:\n    return dt.astimezone(timezone.utc).replace(microsecond=0).isoformat()\n\n\ndef _parse_iso(value: str | None) -> datetime | None:\n    if not value:\n        return None\n    try:\n        return datetime.fromisoformat(value.replace("Z", "+00:00"))\n    except ValueError:\n        return None\n\n\n@dataclass\nclass Skill:\n    """A small curriculum skill descriptor."""\n\n    id: str\n    title: str\n    notebook: str\n    level: int\n    prerequisites: list[str] = field(default_factory=list)\n    tags: list[str] = field(default_factory=list)\n\n    def to_dict(self) -> dict[str, Any]:\n        return asdict(self)\n\n\ndef setup_colab(\n    progress_filename: str = DEFAULT_PROGRESS_FILENAME,\n    mount_drive: bool = True,\n    verbose: bool = True,\n) -> Path:\n    """Prepare a notebook session and return the progress-file path.\n\n    In Colab, this mounts Google Drive when available. Outside Colab, it falls\n    back to the current working directory.\n    """\n\n    progress_path = Path(progress_filename)\n    try:\n        import google.colab  # type: ignore  # noqa: F401\n        in_colab = True\n    except Exception:\n        in_colab = False\n\n    if in_colab and mount_drive:\n        try:\n            from google.colab import drive  # type: ignore\n\n            drive.mount("/content/drive")\n            drive_dir = Path("/content/drive") / DRIVE_PROGRESS_DIR\n            drive_dir.mkdir(parents=True, exist_ok=True)\n            progress_path = drive_dir / progress_filename\n        except Exception as exc:\n            if verbose:\n                print(f"Drive mount skipped; using local progress file. ({exc})")\n\n    if verbose:\n        print(f"Learning environment ready. Progress: {progress_path}")\n    return progress_path\n\n\nclass ProgressStore:\n    """JSON-backed spaced-repetition progress store."""\n\n    def __init__(self, path: str | Path | None = None, learner_id: str = "local"):\n        self.path = Path(path or DEFAULT_PROGRESS_FILENAME)\n        self.learner_id = learner_id\n        self.data = self._load()\n\n    def _default_data(self) -> dict[str, Any]:\n        return {\n            "schema_version": SCHEMA_VERSION,\n            "learner_id": self.learner_id,\n            "skills": {},\n        }\n\n    def _load(self) -> dict[str, Any]:\n        if not self.path.exists():\n            return self._default_data()\n        try:\n            data = json.loads(self.path.read_text(encoding="utf-8"))\n        except Exception:\n            return self._default_data()\n        if data.get("schema_version") != SCHEMA_VERSION:\n            data["schema_version"] = SCHEMA_VERSION\n        data.setdefault("learner_id", self.learner_id)\n        data.setdefault("skills", {})\n        return data\n\n    def save(self) -> None:\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        self.path.write_text(json.dumps(self.data, indent=2), encoding="utf-8")\n\n    def get(self, skill_id: str) -> dict[str, Any]:\n        return self.data.setdefault("skills", {}).setdefault(skill_id, {\n            "attempts": 0,\n            "correct": 0,\n            "confidence": 0,\n            "mastery": 0.0,\n            "last_seen": None,\n            "next_review": None,\n        })\n\n    def record_attempt(\n        self,\n        skill_id: str,\n        correct: bool,\n        confidence: int = 3,\n        item_type: str = "practice",\n    ) -> dict[str, Any]:\n        confidence = max(1, min(5, int(confidence)))\n        row = self.get(skill_id)\n        attempts = int(row.get("attempts", 0)) + 1\n        correct_count = int(row.get("correct", 0)) + int(bool(correct))\n        accuracy = correct_count / attempts\n        confidence_factor = confidence / 5\n        mastery = round((0.75 * accuracy) + (0.25 * confidence_factor), 3)\n\n        row.update({\n            "attempts": attempts,\n            "correct": correct_count,\n            "confidence": confidence,\n            "mastery": mastery,\n            "last_seen": _iso(_utcnow()),\n            "last_item_type": item_type,\n            "next_review": _iso(_utcnow() + review_interval(correct, confidence, attempts)),\n        })\n        self.save()\n        return row\n\n    def review_queue(\n        self,\n        skills: Iterable[Skill | dict[str, Any]] | None = None,\n        limit: int = 8,\n        now: datetime | None = None,\n    ) -> list[dict[str, Any]]:\n        now = now or _utcnow()\n        known_titles = {}\n        if skills:\n            for skill in skills:\n                item = skill.to_dict() if isinstance(skill, Skill) else skill\n                known_titles[item["id"]] = item\n\n        due: list[dict[str, Any]] = []\n        for skill_id, row in self.data.get("skills", {}).items():\n            next_review = _parse_iso(row.get("next_review"))\n            if next_review is None or next_review <= now:\n                due.append({\n                    "id": skill_id,\n                    "title": known_titles.get(skill_id, {}).get("title", skill_id),\n                    "mastery": row.get("mastery", 0.0),\n                    "next_review": row.get("next_review"),\n                    "attempts": row.get("attempts", 0),\n                })\n        due.sort(key=lambda item: (item["mastery"], item["next_review"] or ""))\n        return due[:limit]\n\n    def weak_skills(self, limit: int = 8) -> list[tuple[str, dict[str, Any]]]:\n        rows = list(self.data.get("skills", {}).items())\n        rows.sort(key=lambda kv: (kv[1].get("mastery", 0.0), -kv[1].get("attempts", 0)))\n        return rows[:limit]\n\n\ndef review_interval(correct: bool, confidence: int, attempts: int = 1) -> timedelta:\n    """Return the next review interval from correctness and confidence."""\n\n    if not correct:\n        return timedelta(hours=6) if attempts <= 1 else timedelta(days=1)\n    if confidence <= 2:\n        return timedelta(days=1)\n    if confidence == 3:\n        return timedelta(days=3)\n    high_confidence_days = [7, 14, 30, 60]\n    return timedelta(days=high_confidence_days[min(max(attempts - 1, 0), len(high_confidence_days) - 1)])\n\n\ndef record_attempt(\n    skill_id: str,\n    correct: bool,\n    confidence: int = 3,\n    item_type: str = "practice",\n    store: ProgressStore | None = None,\n) -> dict[str, Any]:\n    store = store or ProgressStore()\n    return store.record_attempt(skill_id, correct, confidence, item_type)\n\n\ndef check(\n    name: str,\n    actual: Any,\n    expected: Any,\n    tolerance: float | None = None,\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Friendly equality/numeric check with optional progress recording."""\n\n    if tolerance is not None and isinstance(actual, (int, float)) and isinstance(expected, (int, float)):\n        ok = math.isclose(actual, expected, rel_tol=tolerance, abs_tol=tolerance)\n    else:\n        ok = actual == expected\n\n    if ok:\n        print(f"PASS: {name}")\n    else:\n        print(f"TRY AGAIN: {name}")\n        print(f"  expected: {expected!r}")\n        print(f"  actual:   {actual!r}")\n\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "check", store)\n    return ok\n\n\ndef code_check(\n    name: str,\n    fn: Callable[..., Any],\n    cases: Iterable[tuple[tuple[Any, ...], Any] | tuple[tuple[Any, ...], dict[str, Any], Any]],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Run multiple function cases and report the first failure."""\n\n    all_ok = True\n    for index, case in enumerate(cases, 1):\n        if len(case) == 2:\n            args, expected = case  # type: ignore[misc]\n            kwargs = {}\n        else:\n            args, kwargs, expected = case  # type: ignore[misc]\n        try:\n            actual = fn(*args, **kwargs)\n            ok = actual == expected\n        except Exception as exc:\n            actual = f"{type(exc).__name__}: {exc}"\n            ok = False\n        if not ok:\n            all_ok = False\n            print(f"TRY AGAIN: {name} case {index}")\n            print(f"  args:     {args}")\n            print(f"  expected: {expected!r}")\n            print(f"  actual:   {actual!r}")\n            break\n    if all_ok:\n        print(f"PASS: {name} ({index} cases)")\n    if skill_id:\n        record_attempt(skill_id, all_ok, confidence, "code_check", store)\n    return all_ok\n\n\ndef normalize_answer(value: Any) -> str:\n    text = str(value).strip().lower()\n    text = re.sub(r"\\s+", " ", text)\n    text = re.sub(r"[^a-z0-9.+/=\\- ]", "", text)\n    text = re.sub(r"\\s*=\\s*", "=", text)\n    return text\n\n\ndef short_answer_check(\n    name: str,\n    answer: Any,\n    accepted: Iterable[Any],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    normalized = normalize_answer(answer)\n    accepted_norm = {normalize_answer(item) for item in accepted}\n    ok = normalized in accepted_norm\n    print(f"{\'PASS\' if ok else \'TRY AGAIN\'}: {name}")\n    if not ok:\n        print("  Re-read the prompt and try to retrieve the answer before opening a hint.")\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "short_answer", store)\n    return ok\n\n\ndef hint_box(title: str, hints: list[str], unlock: bool = False) -> None:\n    """Display staged hints. Full hints require unlock=True."""\n\n    print(title)\n    if not hints:\n        print("No hints for this item.")\n        return\n    if not unlock:\n        print(f"Hint 1: {hints[0]}")\n        print("Run again with unlock=True after a real attempt to see more.")\n        return\n    for index, hint in enumerate(hints, 1):\n        print(f"Hint {index}: {hint}")\n\n\ndef mastery_dashboard(\n    store: ProgressStore | None = None,\n    skills: Iterable[Skill | dict[str, Any]] | None = None,\n    next_notebook: str | None = None,\n) -> None:\n    store = store or ProgressStore()\n    due = store.review_queue(skills=skills)\n    weak = store.weak_skills()\n    total = len(store.data.get("skills", {}))\n    mastered = sum(1 for row in store.data.get("skills", {}).values() if row.get("mastery", 0) >= 0.8)\n\n    print("Mastery Dashboard")\n    print("=================")\n    print(f"Skills attempted: {total}")\n    print(f"Skills at mastery >= 0.80: {mastered}")\n    if due:\n        print("\\nDue for review:")\n        for item in due:\n            print(f"- {item[\'title\']} (mastery {item[\'mastery\']})")\n    else:\n        print("\\nNo due reviews yet.")\n    if weak:\n        print("\\nWeakest skills:")\n        for skill_id, row in weak:\n            print(f"- {skill_id}: mastery {row.get(\'mastery\', 0)} after {row.get(\'attempts\', 0)} attempts")\n    if next_notebook:\n        print(f"\\nRecommended next notebook: {next_notebook}")\n', encoding='utf-8')    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )progress_path = setup_colab()store = ProgressStore(progress_path)print('Ready for retrieval practice.')

In [ ]:
NOTEBOOK = {    "id": "17_tic_tac_toe_connect_four",    "level": 4,    "title": "Algorithms 17: Tic-Tac-Toe",    "prerequisites": [        "Algorithms 01"    ],    "skills_taught": [        "lesson.algorithms_17_tic_tac_toe_connect_four"    ],    "skills_practiced": [        "py.arithmetic.expressions"    ],    "next_notebook": "18_euler_estimation.ipynb"}SKILLS = [    {        "id": "lesson.algorithms_17_tic_tac_toe_connect_four",        "title": "Lesson Algorithms 17 Tic Tac Toe Connect Four",        "notebook": "17_tic_tac_toe_connect_four.ipynb",        "level": 4    },    {        "id": "py.arithmetic.expressions",        "title": "Py Arithmetic Expressions",        "notebook": "17_tic_tac_toe_connect_four.ipynb",        "level": 4    }]

## Before You Learn: Retrieval GateClose notes for a moment. Try to pull prerequisite ideas from memory before continuing. Getting stuck is useful data for the spaced-review system.

In [ ]:
due = store.review_queue(skills=SKILLS, limit=5)if due:    print('Due review items:')    for item in due:        print(f"- {item['title']} (mastery {item['mastery']})")else:    print('No due reviews yet. Use the recall prompt below to warm up.')

**Recall prompt.** Before reading, name one key idea or procedure you remember from: Algorithms 01.

In [ ]:
recall_answer = ''  # write a one-sentence memory before continuingretrieved = len(recall_answer.strip()) >= 12record = store.record_attempt('lesson.algorithms_17_tic_tac_toe_connect_four', retrieved, confidence=3, item_type='prerequisite_recall')print('Recorded retrieval attempt.' if retrieved else 'Write a real memory first, then rerun this cell.')record

## Learning LoopUse the existing lesson cells below as the micro-lesson and practice surface. For each exercise: predict first, attempt from memory, run checks, then open explanations only after the attempt.

In [ ]:
# Google Colab Setupimport math, randomprint('Ready ✅')

---## Phase −1 — Theoretical Foundation### Game Development Architecture> 📖 *From Algorithms Ch. 17:* Develop a tic-tac-toe game in which two random players play against each other.We need separation of concerns:- `Game` Class: Owns the 3x3 board. Knows the rules. Knows who won. Validates moves.- `Player` Class: Receives a *copy* of the board. Decides what move to play. Returns the move.**Crucial Security Rule:**Players should NOT update the board themselves! Otherwise, they could cheat. The Game asks the player for a move, the game verifies it is legal, and the game updates the board. If the move is illegal, the game skips the player's turn.

### Comprehension Check ✅1. Why must the Game pass a *copy* of the board to the Player, rather than the original board array?2. What is the difference between a `RandomPlayer` and a `ManualPlayer` in terms of class design?<details><summary>💡 Explanation after a retrieval attempt</summary>1. Array Referencing trap! If the game passes the original array, the Player can directly mutate it (e.g. `board[0][0] = 'X'`) bypassing all rule checks.2. They share the same interface (they both have a `get_move(board)` method), but `RandomPlayer` uses `random.choice`, while `ManualPlayer` uses Python's `input()` to ask the human.</details>

---## Phase 0 — Math Foundation Practice### 🎯 Your Turn: Win Detection LogicBefore building classes, write the raw logic to check if a 3x3 board (represented as a 2D array of 'X', 'O', or ' ') has a winner.

In [ ]:
def check_winner(board):    '''Return 'X', 'O', or None.'''    # Check 3 rows, 3 cols, 2 diagonals    # YOUR CODE HERE    pass# assert check_winner([['X','X','X'],[' ','O',' '],[' ',' ','O']]) == 'X'# print('Phase 0 passed ✅')

---## Phase 1 — Algorithm Construction### Step 1: The Player ClassesImplement `RandomPlayer` and `ManualPlayer`. They must implement `get_move(board)` and return a tuple `(row, col)`.

In [ ]:
import randomclass RandomPlayer:    def __init__(self, piece):        self.piece = piece # 'X' or 'O'            def get_move(self, board):        # Find all empty spaces ' '        # Return a random one as (row, col)        # YOUR CODE HERE        passclass ManualPlayer:    def __init__(self, piece):        self.piece = piece            def get_move(self, board):        # Print the board nicely        # Ask for input e.g. "Enter row and col (0-2): "        # Return (row, col)        # YOUR CODE HERE        pass

### Step 2: The Game ClassImplement the `Game` class. It takes `player1` and `player2`.Implement `run(self, log=True)` which loops until someone wins or the board is full (tie).

In [ ]:
class TicTacToe:    def __init__(self, player1, player2):        self.board = [[' ' for _ in range(3)] for _ in range(3)]        self.players = [player1, player2]        self.turn = 0            def run(self, log=True):        while True:            current_player = self.players[self.turn % 2]                        # 1. Get move from player (pass a DEEP COPY of self.board!)            # 2. Check if move is legal (bounds check & space is empty)            # 3. If legal, update board. If illegal, print warning and skip turn.            # 4. Check for winner or tie. Break loop if game over.            # 5. self.turn += 1            # YOUR CODE HERE            break # Remove this# Run a simulation!# p1 = RandomPlayer('X')# p2 = RandomPlayer('O')# game = TicTacToe(p1, p2)# game.run(log=True)

---## Phase 2 — Experimental Verification### The Arena (Interleaved Practice)Write a script that runs 1000 games of `RandomPlayer` vs `RandomPlayer` (with logging disabled!).Count the number of times X wins, O wins, and Ties occur. *(Since X goes first, X should win more often than O!)*

In [ ]:
def run_arena(trials=1000):    x_wins = 0    o_wins = 0    ties = 0        # YOUR CODE HERE        print(f"X wins: {x_wins} | O wins: {o_wins} | Ties: {ties}")# run_arena()

---## Phase 3 — Olympiad Extension### Challenge: Connect FourTic-Tac-Toe is a $3 \times 3$ grid needing 3 in a row.Connect Four is a $6 \times 7$ grid needing 4 in a row. Furthermore, gravity exists! Players choose a *column* (0-6), and the piece falls to the lowest available row in that column.Clone your Tic-Tac-Toe game and modify it into Connect Four. The `check_winner` logic will be much harder!

In [ ]:
class ConnectFour:    # YOUR CODE HERE    pass

### Chalkboard DefenseExplain how you would write a `MinimaxPlayer` that never loses at Tic-Tac-Toe. What does the algorithm do conceptually?<details><summary>💡 Sketch after a retrieval attempt</summary>Minimax generates a **Game Tree** of every possible future move. At the bottom of the tree, it scores wins as +1, losses as -1, and ties as 0.It then percolates these scores back up the tree. When it is X's turn, X chooses the move that *maximizes* the score. When it is O's turn, O chooses the move that *minimizes* the score.Because the search space of Tic-Tac-Toe is tiny ($< 362,880$ nodes), Minimax can see the end of the game from turn 1 and play perfectly.</details>---📚 **Next:** Algorithms 18 (Euler Estimation)

## End-of-Notebook ReviewRetrieve the core idea one more time before leaving. This final recall makes the next review easier.

In [ ]:
final_recall = ''  # summarize the most important idea in your own wordsconfidence = 3  # set 1-5 after answeringok = len(final_recall.strip()) >= 20store.record_attempt('lesson.algorithms_17_tic_tac_toe_connect_four', ok, confidence=confidence, item_type='end_review')mastery_dashboard(store=store, skills=SKILLS, next_notebook='18_euler_estimation.ipynb')